In [2]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def analyze_experiments(results_dir="results"):
    all_data = []
    
    # 1. 讀取資料夾內所有 JSON
    for filename in os.listdir(results_dir):
        if filename.endswith("AB") and filename.endswith(".json"):
            with open(os.path.join(results_dir, filename), 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                # 提取關鍵欄位
                n_limit = data["input_config"]["N_limit"]
                results = data["output_results"]
                stats = results["statistics_four_catagories"]
                
                # 這裡假設你的 prior_codes 統計在 JSON 裡只有取消數
                # 為了算百分比，我們需要各類別的總數
                # 如果你的 JSON 裡沒存總數，我們可以用這組實驗的初始統計
                # 這裡示範建立一筆資料
                entry = {
                    "N": n_limit,
                    "Total_Cancelled": results["cancelled_count"],
                    "Drivers_Used": results["drivers_used"],
                    "CP_cancel": stats["CP"],
                    "CO_cancel": stats["CO"],
                    "NP_cancel": stats["NP"],
                    "NO_cancel": stats["NO"]
                }
                all_data.append(entry)

    # 2. 轉換為 DataFrame 並排序
    df = pd.DataFrame(all_data).sort_values("N")
    
    # 3. 繪圖
    plt.figure(figsize=(10, 6))
    categories = ['CP', 'CO', 'NP', 'NO']
    colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4'] # 紅, 橘, 綠, 藍
    
    for cat, color in zip(categories, colors):
        column = f"{cat}_cancel"
        # 這裡我們畫的是取消數量，如果要畫比例，請將 Y 軸改為 (df[column] / 該類別總數)
        plt.plot(df["N"], df[column], marker='o', label=f'Category {cat}', color=color, linewidth=2)
        
        # 在點上標註真實數字
        for i, txt in enumerate(df[column]):
            plt.annotate(txt, (df["N"].iloc[i], df[column].iloc[i]), 
                         textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

    plt.title("Impact of Driver Supply (N) on Trip Cancellations", fontsize=14)
    plt.xlabel("Driver Limit (N)", fontsize=12)
    plt.ylabel("Number of Cancelled Trips", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    
    # 4. 顯示表格數據 (含百分比)
    print("實驗數據摘要表：")
    print(df.to_string(index=False))
    
    plt.tight_layout()
    plt.show()

In [ ]:
analyze_experiments()